### Machine Learning Fundamentals — XGBoost

1. Core Intuition and Boosting Assembly: XGBoost builds an additive ensemble $F_m(x_i) = F_{m-1}(x_i) + \eta \cdot h_m(x_i)$, where $i \in \{1, \dots, N\}$ indexes training samples $x_i$, $m \in \{1, \dots, M\}$ is the current boosting iteration out of $M$ total trees, $F_0(x_i)$ is a baseline constant (e.g., mean target or baseline log-odds), $h_m(x_i)$ is the prediction of the $m$-th decision tree, and $\eta \in (0, 1]$ is the learning rate that scales updates to prevent early overfitting.

2. Generalized Pseudo-Residuals ($g_i$) and Loss Curvature ($h_i$): Optimization relies on the 1st derivative (gradient $g_i$) and 2nd derivative (Hessian $h_i$) of a loss function $L(y_i, F_i)$ comparing target $y_i$ to cumulative prediction $F_i$:
* **Gradient (pseudo-residual):** $g_i = \frac{\partial L(y_i, F_i)}{\partial F_i}$
* **Hessian (loss curvature):** $h_i = \frac{\partial^2 L(y_i, F_i)}{\partial F_i^2}$
* **MSE Loss ($L = \frac{1}{2}(y_i - F_i)^2$):** $g_i = F_i - y_i = -(y_i - F_i)$, $h_i = 1$.
* **Log-Loss ($L = -[y_i\ln p_i + (1-y_i)\ln(1-p_i)]$ where probability $p_i = \sigma(F_i) = \frac{1}{1 + e^{-F_i}}$):** $g_i = p_i - y_i$, $h_i = p_i(1 - p_i)$.

3. Objective Function Derivation (Newton-Raphson Step): At step $m$, expanding loss around current prediction $F_{m-1}(x_i)$ via a 2nd-order Taylor expansion yields:

$$O^{(m)} \approx \sum_{i=1}^N \left[ L(y_i, F_{m-1}(x_i)) + g_i h_m(x_i) + \frac{1}{2} h_i h_m^2(x_i) \right] + \gamma T + \frac{1}{2}\lambda \sum_{j=1}^T w_j^2$$

Where $T$ is the total leaf count of tree $h_m$, $w_j$ is the output weight of leaf $j \in \{1, \dots, T\}$, $\lambda$ is L2 leaf regularization, and $\gamma$ is the minimum split penalty. Dropping constant $L(y_i, F_{m-1}(x_i))$ and grouping sample indices $i$ into their assigned leaf sets $I_j = \{i \mid x_i \in \text{leaf } j\}$ gives:

$$O^{(m)} = \sum_{j=1}^T \left[ \left(\sum_{i \in I_j} g_i\right) w_j + \frac{1}{2} \left(\sum_{i \in I_j} h_i + \lambda\right) w_j^2 \right] + \gamma T$$

4. Exact Optimal Leaf Weight ($w_j^*$) and Similarity Score ($SS_j$): Defining aggregate leaf gradient $G_j = \sum_{i \in I_j} g_i$ and aggregate leaf Hessian $H_j = \sum_{i \in I_j} h_i$, taking derivative $\frac{\partial O^{(m)}}{\partial w_j} = G_j + (H_j + \lambda)w_j = 0$ gives:
* **Optimal Leaf Weight:** $w_j^* = -\frac{G_j}{H_j + \lambda}$
* **Similarity Score ($SS_j$, max loss reduction):** Substituting $w_j^*$ into $O^{(m)}$ yields $SS_j = \frac{G_j^2}{2(H_j + \lambda)}$.

5. Split Gain Formula: For a candidate split dividing a parent node $P$ into Left ($L$) and Right ($R$) child sample sets $I_L$ and $I_R$, the net objective reduction is:

$$\text{Gain} = \frac{1}{2} \left[ \frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L + G_R)^2}{H_L + H_R + \lambda} \right] - \gamma$$

Where $G_L = \sum_{i \in I_L} g_i$, $H_L = \sum_{i \in I_L} h_i$, $G_R = \sum_{i \in I_R} g_i$, $H_R = \sum_{i \in I_R} h_i$, and $G_L + G_R = G_P$, $H_L + H_R = H_P$. A split is executed only if $\text{Gain} > 0$.
6. Node Cover & Min Child Weight: Node Cover is defined as total Hessian sum $\text{Cover}_j = H_j = \sum_{i \in I_j} h_i$. The hyperparameter `min_child_weight` ($M_w$) enforces a threshold constraint $H_L \ge M_w$ and $H_R \ge M_w$ for a split to be valid:
* **In Regression ($h_i = 1$):** $\text{Cover}_j = \vert{}I_j\vert{}$ (exact sample count in leaf $j$).
* **In Classification ($h_i = p_i(1 - p_i)$):** $\text{Cover}_j = \sum_{i \in I_j} p_i(1 - p_i)$ (variance-weighted sample size, where max contribution per sample is $0.25$ at $p_i = 0.5$).

7. Exact Bottom-Up Pruning Execution: Decision trees grow top-down to `max_depth` ($D$). After growth, pruning evaluates nodes bottom-up: if a split yields $\text{Gain} = \frac{1}{2}[SS_L + SS_R - SS_P] - \gamma < 0$, child leaves $L$ and $R$ are pruned, converting the parent back into a single leaf with weight $w_P^* = -\frac{G_P}{H_P + \lambda}$. This test recurses up toward the root.

8. Missing Value Routing (Sparsity-Aware Split): For candidate split threshold $x \le v$, let $I_{\text{mis}} \subset \{1, \dots, N\}$ be the sample subset with missing values ($N/A$) for feature $x$:
* Evaluate $\text{Gain}_{\text{left}}$ with $I_L = \{i \mid x_i \le v\} \cup I_{\text{mis}}$ and $I_R = \{i \mid x_i > v\}$.
* Evaluate $\text{Gain}_{\text{right}}$ with $I_L = \{i \mid x_i \le v\}$ and $I_R = \{i \mid x_i > v\} \cup I_{\text{mis}}$.
* Set default branch direction = $\arg\max(\text{Gain}_{\text{left}}, \text{Gain}_{\text{right}})$. Any unseen missing test sample reaching this node is routed along this default branch.

9. Imbalanced Classification and Prediction Output: Final predicted probability across $M$ trees is $p(x_i) = \sigma(z(x_i)) = \frac{1}{1 + e^{-z(x_i)}}$, where log-odds $z(x_i) = z_0 + \eta \sum_{m=1}^M h_m(x_i)$. To handle positive class imbalance ($y_i=1$), the parameter `scale_pos_weight` ($w_{\text{pos}}$) modifies positive sample derivatives:
* For $y_i=1$: $g_i \leftarrow w_{\text{pos}} \cdot (p_i - 1)$ and $h_i \leftarrow w_{\text{pos}} \cdot p_i(1 - p_i)$.
* For $y_i=0$: $g_i = p_i - 0$ and $h_i = p_i(1 - p_i)$ remain unweighted.
* This scales up $G_j = \sum_{i \in I_j} g_i$, forcing optimal leaf weight $w_j^* = -\frac{G_j}{H_j + \lambda}$ to output larger adjustments for positive class instances.

### Example :

Step-by-Step Worked Example (Binary Classification, $N=4$ samples, $x=[1,2,8,9]$, $y=[0,0,1,1]$, parameters: $\lambda=0, \eta=0.3, \gamma=0$, initial log-odds $z_0=0 \implies p_0 = \frac{1}{1+e^0} = 0.5$):
* **Round 1 Calculations:**
* $g_i = p_0 - y_i \implies g = [0.5, 0.5, -0.5, -0.5]$
* $h_i = p_0(1 - p_0) = 0.5(0.5) \implies h = [0.25, 0.25, 0.25, 0.25]$
* Split test $x \le 5 \implies I_L = \{1,2\}$ (indices where $x_i \le 2$), $I_R = \{3,4\}$ (indices where $x_i \ge 8$)
* $G_L = 0.5 + 0.5 = 1.0, H_L = 0.25 + 0.25 = 0.5 \implies w_L^* = -\frac{1.0}{0.5 + 0} = -2.0$
* $G_R = -0.5 - 0.5 = -1.0, H_R = 0.25 + 0.25 = 0.5 \implies w_R^* = -\frac{-1.0}{0.5 + 0} = +2.0$
* $\text{Gain} = \frac{1}{2}\left[\frac{1.0^2}{0.5} + \frac{(-1.0)^2}{0.5} - \frac{0^2}{1.0}\right] - 0 = 2.0 > 0$
* Log-Odds Update $z_1 = z_0 + \eta w^* \implies z_L = 0 + 0.3(-2.0) = -0.6; z_R = 0 + 0.3(2.0) = +0.6$
* Probability Update $p_i = \sigma(z_1) \implies p_L = \frac{1}{1+e^{0.6}} = 0.354; p_R = \frac{1}{1+e^{-0.6}} = 0.646 \implies p = [0.354, 0.354, 0.646, 0.646]$


* **Round 2 Calculations:**
* $g_i = p_i - y_i \implies g = [0.354, 0.354, -0.354, -0.354]$
* $h_i = p_i(1 - p_i) = 0.354(0.646) \implies h = [0.229, 0.229, 0.229, 0.229]$
* Split $x \le 5 \implies G_L = 0.708, H_L = 0.458 \implies w_L^* = -\frac{0.708}{0.458 + 0} = -1.546$
* $G_R = -0.708, H_R = 0.458 \implies w_R^* = -\frac{-0.708}{0.458 + 0} = +1.546$
* $\text{Gain} = \frac{1}{2}\left[\frac{0.708^2}{0.458} + \frac{(-0.708)^2}{0.458} - 0\right] - 0 = 1.094 > 0$
* Log-Odds Update $z_2 = z_1 + \eta w^* \implies z_L = -0.6 + 0.3(-1.546) = -1.064; z_R = 0.6 + 0.3(1.546) = +1.064$
* Probability Update $p_i = \sigma(z_2) \implies p_L = \frac{1}{1+e^{1.064}} = 0.257; p_R = \frac{1}{1+e^{-1.064}} = 0.743 \implies p = [0.257, 0.257, 0.743, 0.743]$




In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def log_loss(y, p, eps=1e-15):
    p = np.clip(p, eps, 1 - eps)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def gradient(y, p):    # g = p - y
    return p - y

def hessian(p):         # h = p(1-p)
    return p * (1 - p)

print("g:", gradient(np.array([1.0]), np.array([0.8])), "h:", hessian(np.array([0.8])))  # -0.2, 0.16

from sklearn.metrics import log_loss as sk_log_loss
y, z = np.array([1, 0, 1, 1, 0]), np.array([0.5, -1.2, 2.0, 0.1, -0.3])
p = sigmoid(z)
print("log loss (ours vs sklearn):", log_loss(y, p), sk_log_loss(y, p))

In [ ]:
def leaf_score(g, h, lam=0.0):    # G^2 / (H + lambda)
    return (np.sum(g) ** 2) / (np.sum(h) + lam)

def leaf_value(g, h, lam=0.0):    # -G / (H + lambda)
    return -np.sum(g) / (np.sum(h) + lam)

print("leaf_score:", leaf_score(np.array([-0.2, -0.3]), np.array([0.16, 0.21])))  # ~0.676

In [ ]:
def best_split_xgb(X, g, h, lam=0.0):
    best_gain = -float("inf")
    best_feature, best_threshold = None, None
    parent_score = leaf_score(g, h, lam)

    for feature_idx in range(X.shape[1]):
        values = np.unique(X[:, feature_idx])
        thresholds = (values[:-1] + values[1:]) / 2

        for t in thresholds:
            left_mask = X[:, feature_idx] <= t
            g_left, h_left = g[left_mask], h[left_mask]
            g_right, h_right = g[~left_mask], h[~left_mask]

            if len(g_left) == 0 or len(g_right) == 0:
                continue

            gain = leaf_score(g_left, h_left, lam) + leaf_score(g_right, h_right, lam) - parent_score

            if gain > best_gain:
                best_gain = gain
                best_feature, best_threshold = feature_idx, t

    return best_feature, best_threshold, best_gain

In [ ]:
X = np.array([1, 2, 8, 9], dtype=float).reshape(-1, 1)
y_toy = np.array([0, 0, 1, 1], dtype=float)

def boosting_round(X, y, z, lam=0.0, lr=0.3):
    p = sigmoid(z)
    g, h = gradient(y, p), hessian(p)
    feature, threshold, gain = best_split_xgb(X, g, h, lam)
    left_mask = X[:, feature] <= threshold
    left_value = leaf_value(g[left_mask], h[left_mask], lam)
    right_value = leaf_value(g[~left_mask], h[~left_mask], lam)
    tree_output = np.where(left_mask, left_value, right_value)
    return z + lr * tree_output, gain, left_value, right_value

z = np.zeros(4)
for round_num in range(1, 3):
    z, gain, left_val, right_val = boosting_round(X, y_toy, z)
    print(f"round {round_num}: gain={gain:.3f}, values=({left_val:.3f}, {right_val:.3f}), p={np.round(sigmoid(z), 3)}")
# expect: round1 gain=4.0, p=[.354,.354,.646,.646]; round2 gain~2.19, p=[.257,.257,.743,.743]

In [ ]:
from xgboost import XGBClassifier

# approximate check against the real library -- internals differ in minor ways, but
# direction and scale should agree with the worked example above
xgb_toy = XGBClassifier(n_estimators=2, max_depth=1, learning_rate=0.3, reg_lambda=0,
                          base_score=0.5, eval_metric="logloss")
xgb_toy.fit(X, y_toy)
print("xgboost p:", np.round(xgb_toy.predict_proba(X)[:, 1], 3))

In [ ]:
class TreeNode:
    def __init__(self, g, h, lam):
        self.value = leaf_value(g, h, lam)   # computed up front so pruning can collapse back to this
        self.is_leaf = True
        self.feature = self.threshold = self.gain = None
        self.left = self.right = None

def grow_tree(X, g, h, lam=1.0, max_depth=2, depth=0):
    node = TreeNode(g, h, lam)
    if depth >= max_depth or len(g) < 2:
        return node   # only stopping rule during growth is depth/size -- NOT gain
    feature, threshold, gain = best_split_xgb(X, g, h, lam)
    if feature is None:
        return node
    left_mask = X[:, feature] <= threshold
    node.is_leaf = False
    node.feature, node.threshold, node.gain = feature, threshold, gain
    node.left = grow_tree(X[left_mask], g[left_mask], h[left_mask], lam, max_depth, depth + 1)
    node.right = grow_tree(X[~left_mask], g[~left_mask], h[~left_mask], lam, max_depth, depth + 1)
    return node

def prune_tree(node, gamma):
    if node.is_leaf:
        return
    prune_tree(node.left, gamma)    # deepest splits resolved first -- bottom-up
    prune_tree(node.right, gamma)
    if node.left.is_leaf and node.right.is_leaf and (node.gain - gamma) < 0:
        node.is_leaf = True         # collapse back; node.value already computed at construction
        node.left = node.right = None

def predict_tree(node, x_row):
    if node.is_leaf:
        return node.value
    branch = node.left if x_row[node.feature] <= node.threshold else node.right
    return predict_tree(branch, x_row)


z0 = np.zeros(4)
g0, h0 = gradient(y_toy, sigmoid(z0)), hessian(sigmoid(z0))
tree = grow_tree(X, g0, h0, lam=1.0, max_depth=2)   # lam=1 here (not 0) to see regularization act
print(f"root gain={tree.gain:.3f} (expect ~1.333); depth-1 gains={tree.left.gain:.3f},{tree.right.gain:.3f} (expect ~-0.267)")

prune_tree(tree, gamma=0.1)
print(f"root kept: {not tree.is_leaf} (gain-gamma={tree.gain-0.1:.3f} >= 0)")
print(f"leaf values after pruning: {tree.left.value:.3f}, {tree.right.value:.3f} (expect -0.667, +0.667)")

# Cover check: root Sigma h = 1.0 (4 pts at p=0.5, h=0.25 each). min_child_weight>0.5 would
# block the depth-1 split -- each resulting child's cover is exactly 0.5
print("cover(root):", np.sum(h0), "| cover(left child):", np.sum(h0[X[:,0]<=5]))